In [ ]:
%pip install langchain langchain-community langchainhub langchain-chroma langchain-openai langchain-text-splitters langchain-classic beautifulsoup4 yfinance gradio

In [ ]:
import os
import pprint
import getpass
import bs4
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from google.colab import userdata

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import gradio as gr

/tmp/ipykernel_4858/4123518759.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

def getSiteMap(url):
    req = Request(
        url=url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response = urlopen(req)
    xml = BeautifulSoup(
        response,
        "lxml-xml",
        from_encoding=response.info().get_param("charset")
    )
    print(xml)
    return xml


def getUrls(xml, name=None, data=None, verbose=False):
    urls = []
    for loc in xml.find_all("loc"):
            urls.append(loc.text.strip())
    return urls

url="https://docs.djangoproject.com/sitemap.xml"
xml=getSiteMap(url)
urls=getUrls(xml)

docs = []
# Loading just the first 10 for speed; adjust as needed
for i, url in enumerate(urls[:10]):
    loader = WebBaseLoader(url)
    docs.extend(loader.load())
    if i % 1== 0:
        print("Loaded document index:", i)

print("DOCS", docs)

<?xml version="1.0" encoding="utf-8"?>
<sitemapindex xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
<sitemap><loc>https://docs.djangoproject.com/sitemap-el.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-en.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-es.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-fr.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-id.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-it.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-ja.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-ko.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-pl.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-pt-br.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-sv.xml</loc></sitemap><sitemap><loc>https://docs.djangoproject.com/sitemap-zh-hans.xml</

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

print(f"Len docs: {len(docs)}, Len splits: {len(splits)}")

Len docs: 10, Len splits: 2046


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # Updated to newer, cheaper model

system_prompt = (
    "You are a django assistant  for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "If the question is not clear ask follow up questions"
    "\n\n"
    "{context}"
)

# History Aware Retriever
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage

def predict(message, history):
    """
    message: str
    history: list of dicts in OpenAI-style format when type='messages'
             e.g. [{"role":"user","content":"hi"}, {"role":"assistant","content":"hello"}]
    """
    history_for_llm = []
    for item in history:
        role = item.get("role")
        content = item.get("content", "")
        if role == "user":
            history_for_llm.append(HumanMessage(content=content))
        elif role == "assistant":
            history_for_llm.append(AIMessage(content=content))

    result = rag_chain.invoke({"input": message, "chat_history": history_for_llm})
    return result["answer"]

with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# DocumentQABot")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(
        placeholder="Hi! I am your virtual assistant, how can I help you today?",
        container=False,
        scale=7,
    )

    with gr.Row():
        undo = gr.Button("Delete Previous")
        clear = gr.Button("Clear")

    chat = gr.ChatInterface(
        fn=predict,
        chatbot=chatbot,
        textbox=msg,
        examples=["What is Django?", "How to install Django?"],
        title=None,  # already using Markdown title
    )

    # Clear chat
    clear.click(lambda: [], None, chatbot)

    # Undo last turn (removes last 2 messages if present: user+assistant)
    def undo_last(history):
        if not history:
            return []
        # In messages format, history is a list of dicts, typically alternating user/assistant
        return history[:-2] if len(history) >= 2 else []

    undo.click(undo_last, chatbot, chatbot)

demo.launch(share=True, debug=False)

/tmp/ipykernel_4858/2453818297.py:22: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme="soft") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c610b36503c3b99a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
